# Project Contributors

Inês Honrado, Jude Gbenimako, Ruben Marques, Susana Reis, and Tomás Carvalho.

*NOVA IMS — Machine Learning Group Project (2024/2025).*

# Import the Necessary Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from scipy.stats import skew
from sklearn.impute import SimpleImputer

from sklearn.preprocessing import LabelEncoder
import category_encoders as ce
from sklearn.preprocessing import MinMaxScaler

from scipy import stats
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LassoCV

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier

from sklearn.metrics import f1_score
import time
from sklearn.model_selection import GridSearchCV

import joblib

import warnings
warnings.filterwarnings('ignore')

# Context

Workers' compensation for workplace injuries in New York are administrated through the Workers' Compensation Board. Due to the fact that over 5 million claims have been processed since the year 2000, assessing claims manually is very time-consuming. The purpose of this project is to apply machine learning models, taking claims from 2020 to 2022, to conduct automated claim evaluations in order to seep up decision-making on new claims and increase WCB efficiency in handling compensation processes.

# Metadata

- **Accident Date**: Injury date of the claim.
- **Age at Injury**: Age of injured worker when the injury occurred.
- **Alternative Dispute Resolution**: Adjudication processes external to the Board.
- **Assembly Date**: The date the claim was first assembled.
- **Attorney/Representative**: Whether the claim is being represented by an attorney.
- **Average Weekly Wage**: The wage used to calculate workers’ compensation, disability, or paid leave wage replacement benefits.
- **Birth Year**: The reported year of birth of the injured worker.
- **C-2 Date**: Date of receipt of the Employer's Report of Work-Related Injury/Illness or equivalent (formerly Form C-2).
- **C-3 Date**: Date Form C-3 (Employee Claim Form) was received.
- **Carrier Name**: Name of the primary insurance provider responsible for providing workers’ compensation coverage to the injured worker’s employer.
- **Carrier Type**: Type of primary insurance provider responsible for providing workers’ compensation coverage.
- **Claim Identifier**: Unique identifier for each claim, assigned by WCB.
- **County of Injury**: Name of the New York County where the injury occurred.
- **COVID-19 Indicator**: Indicates if the claim may be associated with COVID-19.
- **District Name**: Name of the WCB district office that oversees claims for that region or area of the state.
- **First Hearing Date**: Date the first hearing was held on a claim at a WCB hearing location. A blank date means the claim has not yet had a hearing.
- **Gender**: The reported gender of the injured worker.
- **IME-4 Count**: Number of IME-4 forms received per claim. The IME-4 form is the "Independent Examiner's Report of Independent Medical Examination."
- **Industry Code**: NAICS code and descriptions are available at [NAICS Search](https://www.naics.com/search-naics-codes-by-industry/).
- **Industry Code Description**: 2-digit NAICS industry code description used to classify businesses according to their economic activity.
- **Medical Fee Region**: Approximate region where the injured worker would receive medical services.
- **OIICS Nature of Injury Description**: The OIICS nature of injury codes & descriptions are available at [BLS OIICS Manual](https://www.bls.gov/iif/oiics_manual_2007.pdf).
- **WCIO Cause of Injury Code**: The WCIO cause of injury codes & descriptions are available at [WCIO Cause Table](https://www.wcio.org/Active%20PNC/WCIO_Cause_Table.pdf).
- **WCIO Cause of Injury Description**: See description of field above.
- **WCIO Nature of Injury Code**: The WCIO nature of injury codes & descriptions are available at [WCIO Nature Table](https://www.wcio.org/Active%20PNC/WCIO_Nature_Table.pdf).
- **WCIO Nature of Injury Description**: See description of field above.
- **WCIO Part of Body Code**: The WCIO part of body codes & descriptions are available at [WCIO Part Table](https://www.wcio.org/Active%20PNC/WCIO_Part_Table.pdf).
- **WCIO Part of Body Description**: See description of field above.
- **Zip Code**: The reported ZIP code of the injured worker’s home address.
- **Agreement Reached**: Binary variable: Yes if there is an agreement without the involvement of the WCB (unknown at the start of a claim).
- **WCB Decision**: Multiclass variable: Decision of the WCB relative to the claim: "Accident" means that the claim refers to a workplace accident; "Occupational Disease" means illness from the workplace; requires WCB deliberation, so it is unknown at the start of a claim.
- **Claim Injury Type**: Main target variable: Deliberation of the WCB relative to benefits awarded to the claim. Numbering indicates severity.

# Import the data

In [ ]:
X_train_preprocessed = pd.read_csv('X_train_preprocessed.csv')
X_val_preprocessed = pd.read_csv('X_val_preprocessed.csv')
df_test_preprocessed = pd.read_csv('df_test_preprocessed.csv')

X_train_enc_scal = pd.read_csv('X_train_enc_scal.csv')
X_val_enc_scal = pd.read_csv('X_val_enc_scal.csv')
df_test_enc_scal = pd.read_csv('df_test_enc_scal.csv')

y_train_enc = pd.read_csv('y_train_enc.csv')
y_val_enc = pd.read_csv('y_val_enc.csv')

# Model Assessment and Selection

## Feature Selection

Since we are trying to build a model to predict future claims, no date variables will be kept, as those values are specific to the train claims.

In [ ]:
columns_to_remove = ['Accident_ymd', 'Assembly_ymd', 'C2_ymd']

X_train_enc_scal.drop(columns=columns_to_remove, inplace=True)
X_val_enc_scal.drop(columns=columns_to_remove, inplace=True)

In [ ]:
categorical_variables = [
    'Carrier Name', 'Carrier Type', 'County of Injury', 'Industry Code', 
    'WCIO Cause of Injury Code', 'WCIO Nature of Injury Code', 
    'WCIO Part Of Body Code', 'District Name', 
    'Gender', 'Medical Fee Region', 'Zip Code', 'Injury Day of Week'
]

numerical_variables = [
    'Alternative Dispute Resolution', 'Attorney/Representative',
    'Age at Injury', 'Birth Year', 'IME-4 Count',
    'Agreement Reached', 'Number of Dependents', 
    'C-3 Received', 'First Hearing Happened', 'COVID-19 Indicator', 
    'Time to Assembly', 'Time to C-2', 
    'Multiple Form Submissions', 'Above County Wage Average',
    'Average Weekly Wage', 'Wage/Dependents Ratio'
]

### Spearman Correlations

In [ ]:
def cor_heatmap(cor, title='Correlation Heatmap', cmap='Reds'):
    mask = np.triu(np.ones_like(cor, dtype=bool))

    plt.figure(figsize=(18, 16))
    sns.heatmap(data=cor, annot=True, cmap=cmap, fmt='.2f', 
                 mask=mask, linewidths=0.5, linecolor='black', square=True,
                 cbar_kws={"shrink": .8}, annot_kws={"size": 10})  # Adjust annotation size here
    
    plt.title(title, fontsize=20)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

cor_spearman = X_train_enc_scal[numerical_variables].corr(method='spearman')
cor_heatmap(cor_spearman)

High correlations:
- Birth Year and Age at Injury
- Average Weekly Wage and Above County Wage Average
- IME-4 Count and Multiple Form Submissions
- Number of Dependents and Wage/Dependents Ratio
- Time to Assembly and Time to C-2

### RandomForestClassifier Feature Importance

In [ ]:
model = RandomForestClassifier(n_estimators=50, max_depth=9, min_samples_split=50, max_features=None, class_weight='balanced', random_state=15)
model.fit(X_train_enc_scal, y_train_enc)

importances = model.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': X_train_enc_scal.columns, 'Importance': importances})

feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

print(feature_importance_df)

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'])
plt.title('Feature Importance from Random Forest Model')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

We will set a threshold where any feature with importance greater than 2% is considered relevant, as half of the features achieve that value.

### Mutual Information

In [ ]:
mi_scores = mutual_info_classif(X_train_enc_scal, y_train_enc)

mi_df = pd.DataFrame({'Feature': X_train_enc_scal.columns, 'MI Score': mi_scores})
mi_df = mi_df.sort_values(by='MI Score', ascending=False).reset_index(drop=True)
print("Mutual Information Scores:\n", mi_df)

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(mi_df['Feature'], mi_df['MI Score'])
plt.title('Mutual Information Scores')
plt.xlabel('Mutual Information')
plt.ylabel('Feature')
plt.show()

Here, we will mantain the 2% threshold.

### Lasso Regression

In [ ]:
def plot_importance(coef,name):
    imp_coef = coef.sort_values()
    plt.figure(figsize=(8,10))
    imp_coef.plot(kind = "barh")
    plt.title("Feature importance using " + name + " Model")
    plt.show()

In [ ]:
reg = LassoCV(alphas=np.logspace(-4, 0, 100), cv=5)

In [ ]:
reg.fit(X_train_enc_scal, y_train_enc)

In [ ]:
coef = pd.Series(reg.coef_, index=X_train_enc_scal.columns)
coef

In [ ]:
print("Lasso picked " + str(sum(coef != 0)) + " variables and eliminated the other " +  str(sum(coef == 0)) + " variables")

In [ ]:
plot_importance(coef,'Lasso')

## Conclusions

Applying a voting system, and taking into account the 2% threshold for RandomForestClassifier Feature Importances and for Mutual Information, we get to the following conclusions:

| Predictor | Spearman  | RandomForestClassifier Feature Importance | Mutual Information | LASSO Regression | What to do? (One possible way to "solve") |
|---|---|---|---|---|---|
| Age at Injury | High correlated with Birth Year | Discard | Discard | Keep | Discard |
| Alternative Dispute Resolution | Keep | Discard | Discard | Keep | Discard |
| Attorney/Representative | Keep? | Keep | Keep | Keep | Include in the model |
| Average Weekly Wage | High correlated with Above County Wage Average | Keep | Keep | Keep | Include in the model |
| Birth Year | High correlated with Birth Year | Keep | Discard | Keep | Include in the model |
| Carrier Name | NA | Keep | Keep | Keep | Include in the model |
| Carrier Type | NA | Discard | Keep | Keep | Include in the model |
| County of Injury | NA | Discard | Discard | Keep | Discard |
| COVID-19 Indicator | Keep | Discard | Discard | Keep | Discard |
| District Name | NA | Discard | Discard | Keep | Discard |
| Gender | NA | Discard | Keep | Keep | Discard |
| IME-4 Count | High correlated with Multiple Form Submissions | Keep | Keep | Keep | Include in the model |
| Industry Code | NA | Discard | Keep | Keep | Include in the model |
| Medical Fee Region | NA | Discard | Keep | Keep | Include in the model |
| WCIO Cause of Injury Code | NA | Keep | Keep | Keep | Include in the model |
| WCIO Nature of Injury Code | NA | Keep | Keep | Keep | Include in the model |
| WCIO Part Of Body Code | NA | Keep | Keep | Keep | Include in the model |
| Zip Code | NA | Discard | Keep | Keep | Include in the model |
| Agreement Reached | Keep | Keep | Keep | Keep | Include in the model |
| Number of Dependents | High correlated with Wage/Dependents Ratio | Discard | Discard | Discard | Discard |
| C-3 Received | Keep | Keep | Keep | Keep | Include in the model |
| First Hearing Happened | Keep | Keep | Keep | Keep | Include in the model |
| Injury Day of Week | NA | Discard | Discard | Keep | Discard |
| Time to Assembly | High correlated with Time to C-2 | Keep | Keep | Keep | Include in the model |
| Time to C-2 | High correlated with Time to Assembly | Keep | Discard | Keep | Discard (already kept Time to Assembly) |
| Wage/Dependents Ratio | High correlated with Number of Dependents| Discard | Keep | Discard | Discard |
| Multiple Form Submissions | High correlated with IME-4 Count | Discard | Keep | Keep | Discard (already kept IME-4 Count) |
| Above County Wage Average | High correlated with Average Weekly Wage | Discard | Keep | Keep | Discard (already kept Average Weekly Wage) |

In [ ]:
include_variables = [
    'Attorney/Representative',
    'Average Weekly Wage',
    'Birth Year',
    'Carrier Name',
    'Carrier Type',
    'IME-4 Count',
    'Industry Code',
    'Medical Fee Region',
    'WCIO Cause of Injury Code',
    'WCIO Nature of Injury Code',
    'WCIO Part Of Body Code',
    'Zip Code',
    'Agreement Reached',
    'C-3 Received',
    'First Hearing Happened',
    'Time to Assembly'
]

## Model Comparison and Selection

We opted to use the holdout method for our model validation for practical reasons. Given the large size of our dataset, the holdout method was well-suited as it efficiently splits the data into training and validation sets without the additional computational complexity associated with cross-validation. We acknowledge that cross-validation can provide more robust insights by using multiple train-test splits, but its increased computational demands made the holdout method a more feasible choice in our context.

We will use F1 Weighted as our evaluation metric because it balances precision and recall while reflecting the actual class distribution and avoiding over-representing minor classes that may have limited impact on the overall system performance. This ensures a more accurate and fair assessment of the model's performance across all target classes, especially in imbalanced datasets like ours.

In [ ]:
X_train = X_train_enc_scal[include_variables]

In [ ]:
X_val = X_val_enc_scal[include_variables]

### RandomForestClassifier

In [ ]:
def validation_rf(X_train, y_train, X_val, y_val):
    model = RandomForestClassifier(
        n_estimators=50, 
        max_depth=9, 
        min_samples_split=50, 
        max_features=None, 
        random_state=15
    )
    start_time = time.time()
    model.fit(X_train, y_train)
    elapsed_time = time.time() - start_time

    y_val_pred = model.predict(X_val)
    y_train_pred = model.predict(X_train)
    
    val_f1 = f1_score(y_val, y_val_pred, average='weighted')
    
    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    
    return {
        'train_f1': train_f1,
        'val_f1': val_f1,
        'elapsed_time': elapsed_time
    }

In [ ]:
results_rf = validation_rf(X_train, y_train_enc, X_val, y_val_enc)

In [ ]:
results_rf

### DecisionTreeClassifier

In [ ]:
def validation_dt(X_train, y_train, X_val, y_val):
    model = DecisionTreeClassifier(
        max_depth=9, 
        min_samples_split=50, 
        random_state=15
    )
    start_time = time.time()
    model.fit(X_train, y_train)
    elapsed_time = time.time() - start_time
    
    y_val_pred = model.predict(X_val)
    
    y_train_pred = model.predict(X_train)
    
    val_f1 = f1_score(y_val, y_val_pred, average='weighted')
    
    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    
    return {
        'train_f1': train_f1,
        'val_f1': val_f1,
        'elapsed_time': elapsed_time
    }

In [ ]:
results_dt = validation_dt(X_train, y_train_enc, X_val, y_val_enc)

In [ ]:
results_dt

### HistGradientBoostingClassifier

In [ ]:
def validation_hgb(X_train, y_train, X_val, y_val):
    model = HistGradientBoostingClassifier(
        max_iter=2000, 
        max_depth=10, 
        random_state=15
    )
    start_time = time.time()
    model.fit(X_train, y_train)
    elapsed_time = time.time() - start_time
    
    y_val_pred = model.predict(X_val)
    
    y_train_pred = model.predict(X_train)
    
    val_f1 = f1_score(y_val, y_val_pred, average='weighted')
    
    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    
    return {
        'train_f1': train_f1,
        'val_f1': val_f1,
        'elapsed_time': elapsed_time
    }

In [ ]:
results_hgb = validation_hgb(X_train, y_train_enc, X_val, y_val_enc)

In [ ]:
results_hgb

### MLPClassifier

In [ ]:
def validation_mlp(X_train, y_train, X_val, y_val):
    model = MLPClassifier(
        solver='adam',
        hidden_layer_sizes=(X_train.shape[1] * 2, X_train.shape[1] * 2, X_train.shape[1] * 2),
        activation='relu',
        learning_rate_init=0.0025,
        max_iter=1000,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=20,
        tol=1e-6
    )
    
    start_time = time.time()
    model.fit(X_train, y_train)
    elapsed_time = time.time() - start_time
    
    y_val_pred = model.predict(X_val)
    
    y_train_pred = model.predict(X_train)
    
    val_f1 = f1_score(y_val, y_val_pred, average='weighted')

    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    
    return {
        'train_f1': train_f1,
        'val_f1': val_f1,
        'elapsed_time': elapsed_time
    }

In [ ]:
results_mlp = validation_mlp(X_train, y_train_enc, X_val, y_val_enc)

In [ ]:
results_mlp

### AdaBoostClassifier

In [ ]:
def validation_ab(X_train, y_train, X_val, y_val):

    base_model = DecisionTreeClassifier(
        max_depth=9, 
        min_samples_split=50, 
        random_state=15
    )
    
    model = AdaBoostClassifier(
        estimator=base_model, 
        n_estimators=50, 
        random_state=15
    )
    
    start_time = time.time()
    model.fit(X_train, y_train)
    elapsed_time = time.time() - start_time
    
    y_val_pred = model.predict(X_val)
    
    y_train_pred = model.predict(X_train)
    
    val_f1 = f1_score(y_val, y_val_pred, average='weighted')
    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    
    return {
        'train_f1': train_f1,
        'val_f1': val_f1,
        'elapsed_time': elapsed_time
    }

In [ ]:
results_ab = validation_ab(X_train, y_train_enc, X_val, y_val_enc)

In [ ]:
results_ab

### Voting Classifier

In [ ]:
def validation_vc(X_train, y_train, X_val, y_val):

    rf_model = RandomForestClassifier(
        n_estimators=50, 
        max_depth=9, 
        min_samples_split=50, 
        random_state=15
    )

    dt_model = DecisionTreeClassifier(
        max_depth=9, 
        min_samples_split=50, 
        random_state=15
    )

    hgb_model = HistGradientBoostingClassifier(
        max_iter=2000, 
        max_depth=10, 
        random_state=15
    )

    mlp_model = MLPClassifier(
        solver='adam',
        hidden_layer_sizes=(X_train.shape[1] * 2, X_train.shape[1] * 2, X_train.shape[1] * 2),
        activation='relu',
        learning_rate_init=0.0025,
        max_iter=1000,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=20,
        tol=1e-6
    )
    
    model = VotingClassifier(
        estimators=[
            ('rf', rf_model),
            ('dt', dt_model),
            ('hgb', hgb_model),
            ('mlp', mlp_model)
        ],
        voting='soft'  
    )
    
    start_time = time.time()
    model.fit(X_train, y_train)
    elapsed_time = time.time() - start_time
    
    y_val_pred = model.predict(X_val)
    
    y_train_pred = model.predict(X_train)
    
    val_f1 = f1_score(y_val, y_val_pred, average='weighted')
    
    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    
    return {
        'train_f1': train_f1,
        'val_f1': val_f1,
        'elapsed_time': elapsed_time
    }

In [ ]:
results_vc = validation_vc(X_train, y_train_enc, X_val, y_val_enc)

In [ ]:
results_vc

### Stacking Classifier

In [ ]:
def validation_stack(X_train, y_train, X_val, y_val):
    rf_model = RandomForestClassifier(
        n_estimators=50, 
        max_depth=9, 
        min_samples_split=50, 
        random_state=15
    )
    
    hgb_model = HistGradientBoostingClassifier(
        max_iter=2000, 
        max_depth=10, 
        random_state=15
    )
    
    estimators = [
        ('rf', rf_model),
        ('hgbc', hgb_model)
    ]
    
    final_estimator = MLPClassifier(
        solver='adam',
        hidden_layer_sizes=(X_train.shape[1] * 2, X_train.shape[1] * 2, X_train.shape[1] * 2),
        activation='relu',
        learning_rate_init=0.0025,
        max_iter=1000,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=20,
        tol=1e-6
    )
    
    model = StackingClassifier(
        estimators=estimators,
        final_estimator=final_estimator
    )

    
    start_time = time.time()
    model.fit(X_train, y_train)
    elapsed_time = time.time() - start_time
    
    y_val_pred = model.predict(X_val)
    
    y_train_pred = model.predict(X_train)
    
    val_f1 = f1_score(y_val, y_val_pred, average='weighted')
    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    
    return {
        'train_f1': train_f1,
        'val_f1': val_f1,
        'elapsed_time': elapsed_time
    }

In [ ]:
results_stack = validation_stack(X_train, y_train_enc, X_val, y_val_enc)

In [ ]:
results_stack

### Comparing Models

In [ ]:
def create_summary_table(results_rf, results_dt, results_hgb, results_mlp, results_ab, results_vc, results_stack):
    results = []
    models = [
        ("Random Forest", results_rf),
        ("Decision Tree", results_dt),
        ("Hist Gradient Boosting", results_hgb),
        ("MLP Classifier", results_mlp),
        ("Ada Boost Classifier", results_ab),
        ("Voting Classifier", results_vc),
        ("Stacking Classifier", results_stack)
    ]

    for model_name, model_results in models:
        train_f1 = model_results['train_f1']
        val_f1 = model_results['val_f1']
        elapsed_time = model_results['elapsed_time']
        
        results.append({
            "Model": model_name,
            "Train F1 Score": train_f1,
            "Validation F1 Score": val_f1,
            "Time Taken (s)": elapsed_time
        })
    
    summary_table = pd.DataFrame(results)
    
    return summary_table

summary_table = create_summary_table(results_rf, results_dt, results_hgb, results_mlp, results_ab, results_vc, results_stack)
summary_table

From the results, the **Stacking Classifier** achieved the highest performance, outperforming other models in both training and validation F1 scores. 


In the next phase, we will focus on fine-tuning the Stacking Classifie.


## Model Optimization

Since Stacking combines multiple models, we will try to improve each individual model by optimizing their hyperparameters using GridSearchCV.

### RandomForestClassifier GridSearchCV

In [ ]:
rf_model = RandomForestClassifier(
    min_samples_split=50, 
    random_state=15
)

param_grid = {
    'n_estimators': [40, 50, 60],   
    'max_depth': [9, 13, 15]         
}

grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=3, n_jobs=-1, verbose=1)

grid_search.fit(X_train, y_train_enc)

print("Best Parameters found: ", grid_search.best_params_)
print("Best Score found: ", grid_search.best_score_)

### HistGradientBoostingClassifier GridSearchCV

In [ ]:
hgb_model = HistGradientBoostingClassifier(
    random_state=15
)

param_grid = {
    'max_depth': [9, 13, 15],         
    'max_iter': [500, 1000, 2000]     
}

grid_search_hgb = GridSearchCV(estimator=hgb_model, param_grid=param_grid, cv=3, n_jobs=-1, verbose=1)

grid_search_hgb.fit(X_train, y_train_enc)

print("Best Parameters found: ", grid_search_hgb.best_params_)
print("Best Score found: ", grid_search_hgb.best_score_)

### MLPClassifier GridSearchCV

In [ ]:
mlp_model = MLPClassifier(
    solver='adam',
    hidden_layer_sizes=(X_train.shape[1] * 2, X_train.shape[1] * 2, X_train.shape[1] * 2),
    activation='relu',
    max_iter=1000,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=20,
    tol=1e-6
)

param_grid = {
    'learning_rate_init': [0.001, 0.0025]  
}

grid_search_mlp = GridSearchCV(estimator=mlp_model, param_grid=param_grid, cv=2, n_jobs=-1, verbose=1)

grid_search_mlp.fit(X_train, y_train_enc)

print("Best Parameters found: ", grid_search_mlp.best_params_)
print("Best Score found: ", grid_search_mlp.best_score_)

### Optimized StackingClassifier

In [ ]:
def validation_stack_optimized(X_train, y_train, X_val, y_val):
    rf_model = RandomForestClassifier(
        n_estimators=60, 
        max_depth=15, 
        min_samples_split=50, 
        random_state=15
    )
    
    hgb_model = HistGradientBoostingClassifier(
        max_iter=500, 
        max_depth=15, 
        random_state=15
    )
    
    estimators = [
        ('rf', rf_model),
        ('hgbc', hgb_model)
    ]
    
    final_estimator = MLPClassifier(
        solver='adam',
        hidden_layer_sizes=(X_train.shape[1] * 2, X_train.shape[1] * 2, X_train.shape[1] * 2),
        activation='relu',
        learning_rate_init=0.0025,
        max_iter=1000,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=20,
        tol=1e-6
    )
    
    model = StackingClassifier(
        estimators=estimators,
        final_estimator=final_estimator
    )

    
    start_time = time.time()
    model.fit(X_train, y_train)
    elapsed_time = time.time() - start_time
    
    y_val_pred = model.predict(X_val)
    
    y_train_pred = model.predict(X_train)
    
    val_f1 = f1_score(y_val, y_val_pred, average='weighted')

    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    
    return {
        'train_f1': train_f1,
        'val_f1': val_f1,
        'elapsed_time': elapsed_time
    }

In [ ]:
results_stack_optimized = validation_stack_optimized(X_train, y_train_enc, X_val, y_val_enc)

In [ ]:
results_stack_optimized

Although the improvements were not substantial, the improved model still outperformed the previous models. 

## Final Model

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=60, 
    max_depth=15, 
    min_samples_split=50, 
    random_state=15
)
    
hgb_model = HistGradientBoostingClassifier(
    max_iter=500, 
    max_depth=15, 
    random_state=15
)
    
estimators = [
    ('rf', rf_model),
    ('hgbc', hgb_model)
]
    
final_estimator = MLPClassifier(
    solver='adam',
    hidden_layer_sizes=(X_train.shape[1] * 2, X_train.shape[1] * 2, X_train.shape[1] * 2),
    activation='relu',
    learning_rate_init=0.0025,
    max_iter=1000,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=20,
    tol=1e-6
)
    
final_model = StackingClassifier(
    estimators=estimators,
    final_estimator=final_estimator
    )
    
final_model.fit(X_train, y_train_enc)

# Agreement Reached Model

The **df_test** dataset did not contain the column **'Agreement Reached'**, which was essential for our model. To address this, we decided to build a separate model to predict the **'Agreement Reached'** column based on the available features in the test dataset.

In [ ]:
target_ar = 'Agreement Reached'
X_train_ar = X_train_preprocessed.drop(columns=[target_ar])
y_train_ar = X_train_preprocessed[target_ar]

X_val_ar = X_val_preprocessed.drop(columns=[target_ar])
y_val_ar = X_val_preprocessed[target_ar]

df_test_ar = df_test_preprocessed

In [ ]:
y_train_ar = pd.Series(y_train_ar)
y_val_ar = pd.Series(y_val_ar)

In [ ]:
y_train_ar.value_counts()

## Encoding

In [ ]:
cols_to_enc = [
    'Carrier Name', 'Carrier Type', 'County of Injury', 'Industry Code', 
    'WCIO Cause of Injury Code', 'WCIO Nature of Injury Code', 
    'WCIO Part Of Body Code', 'District Name', 
    'Gender', 'Medical Fee Region', 'Zip Code', 'Injury Day of Week'
] 

In [ ]:
target_encoder = ce.TargetEncoder(cols=cols_to_enc)

X_train_ar[cols_to_enc] = target_encoder.fit_transform(X_train_ar[cols_to_enc], y_train_ar)
X_val_ar[cols_to_enc] = target_encoder.transform(X_val_ar[cols_to_enc])
df_test_ar[cols_to_enc] = target_encoder.transform(df_test_ar[cols_to_enc])

## Scaling

In [ ]:
columns=X_train_ar.columns

In [ ]:
scaler = MinMaxScaler()
X_train_ar = scaler.fit_transform(X_train_ar)

In [ ]:
X_val_ar = scaler.transform(X_val_ar)
df_test_ar = scaler.transform(df_test_ar)

In [ ]:
X_train_ar = pd.DataFrame(X_train_ar, columns=columns)
X_val_ar= pd.DataFrame(X_val_ar, columns=columns)
df_test_ar = pd.DataFrame(df_test_ar, columns=columns)

## Feature Selection

### Spearman Correlation

In [ ]:
num_ar = [
    'Alternative Dispute Resolution', 'Attorney/Representative',
    'Age at Injury', 'Birth Year', 'IME-4 Count',
    'Number of Dependents', 'C-3 Received', 'First Hearing Happened', 
    'COVID-19 Indicator', 'Time to Assembly', 'Time to C-2', 
    'Multiple Form Submissions', 'Above County Wage Average',
    'Average Weekly Wage', 'Wage/Dependents Ratio',
    'Accident_ymd', 'Assembly_ymd', 'C2_ymd'
]

In [ ]:
def cor_heatmap(cor, title='Correlation Heatmap', cmap='Reds'):
    mask = np.triu(np.ones_like(cor, dtype=bool))

    plt.figure(figsize=(12, 10))
    sns.heatmap(data=cor, annot=True, cmap=cmap, fmt='.2f', 
                 mask=mask, linewidths=0.5, linecolor='black', square=True,
                 cbar_kws={"shrink": .8})
    
    plt.title(title, fontsize=16)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

cor_spearman_ar = X_train_ar[num_ar].corr(method='spearman')

cor_heatmap(cor_spearman_ar, title='Spearman Correlation Heatmap for Assembly Reached', cmap='Blues')

### RandomForestClassifier Feature Importances

In [ ]:
model_ar = RandomForestClassifier(n_estimators=50, max_depth=9, min_samples_split=50, max_features=None, class_weight='balanced', random_state=15)
model_ar.fit(X_train_ar, y_train_ar)

importances_ar = model_ar.feature_importances_

feature_importance_df_ar = pd.DataFrame({'Feature': X_train_ar.columns, 'Importance': importances_ar})

feature_importance_df_ar = feature_importance_df_ar.sort_values(by='Importance', ascending=False)

print(feature_importance_df_ar)

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df_ar['Feature'], feature_importance_df_ar['Importance'])
plt.title('Feature Importance from Random Forest Model')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

In this case, we will set a threshold where any feature with importance greater than 0.5% is considered relevant. 

### Mutual Information

In [ ]:
mi_scores = mutual_info_classif(X_train_ar, y_train_ar)

mi_df = pd.DataFrame({'Feature': X_train_ar.columns, 'MI Score': mi_scores})
mi_df = mi_df.sort_values(by='MI Score', ascending=False).reset_index(drop=True)
print("Mutual Information Scores:\n", mi_df)

For Mutual Information, we will keep the 2% thresold.

### Lasso Regression

In [ ]:
def plot_importance(coef,name):
    imp_coef = coef.sort_values()
    plt.figure(figsize=(8,10))
    imp_coef.plot(kind = "barh")
    plt.title("Feature importance using " + name + " Model")
    plt.show()

reg = LassoCV(alphas=np.logspace(-4, 0, 100), cv=5)

reg.fit(X_train_ar, y_train_ar)

In [ ]:
coef = pd.Series(reg.coef_, index=X_train_ar.columns)
coef

In [ ]:
print("Lasso picked " + str(sum(coef != 0)) + " variables and eliminated the other " +  str(sum(coef == 0)) + " variables")

In [ ]:
plot_importance(coef,'Lasso')

### Conclusions

Applying a voting system, and taking into account the 0.05% threshold for RandomForestClassifier Feature Importances and the 2% for Mutual Information, we get to the following conclusions:

| Predictor | Spearman  | RandomForestClassifier Feature Importance | Mutual Information | LASSO Regression | What to do? (One possible way to "solve") |
|---|---|---|---|---|---|
| Age at Injury | High correlated with Birth Year | Discard | Discard | Keep | Discard |
| Alternative Dispute Resolution | Keep | Discard | Discard | Discard | Discard |
| Attorney/Representative | Keep | Keep | Keep | Keep | Include in the model |
| Average Weekly Wage | High correlated with Above County Wage Average | Keep | Keep | Keep | Include in the model |
| Birth Year | High correlated with Birth Year | Discard | Discard | Keep | Discard |
| Carrier Name | NA | Keep | Keep | Keep | Include in the model |
| Carrier Type | NA | Discard | Keep | Keep | Include in the model |
| County of Injury | NA | Discard | Discard | Keep | Discard |
| COVID-19 Indicator | Keep | Discard | Discard | Discard | Discard |
| District Name | NA | Discard | Keep | Keep | Include in the model |
| Gender | NA | Discard | Keep | Discard | Discard |
| IME-4 Count | High correlated with Multiple Form Submissions | Keep | Keep | Keep | Include in the model |
| Industry Code | NA | Keep | Discard | Keep | Include in the model |
| Medical Fee Region | NA | Discard | Keep | Keep | Include in the model |
| WCIO Cause of Injury Code | NA | Discard | Discard | Keep | Discard |
| WCIO Nature of Injury Code | NA | Discard | Keep | Keep | Include in the model |
| WCIO Part Of Body Code | NA | Discard | Discard | Keep | Discard |
| Zip Code | NA | Keep | Discard | Keep | Include in the model |
| Number of Dependents | High correlated with Wage/Dependents Ratio | Discard | Discard | Discard | Discard |
| C-3 Received | Keep | Discard | Keep | Keep | Include in the model |
| First Hearing Happened | Keep | Keep | Keep | Keep | Include in the model |
| Injury Day of Week | NA | Keep | Discard | Keep | Include in the model |
| Time to Assembly | High correlated with Time to C-2 | Discard | Discard | Keep | Discard |
| Time to C-2 | High correlated with Time to Assembly | Keep | Discard | Discard | Discard |
| Wage/Dependents Ratio | High correlated with Number of Dependents| Discard | Discard | Discard | Discard |
| Multiple Form Submissions | High correlated with IME-4 Count | Discard | Keep | Keep | Discard (already kept IME-4 Count) |
| Above County Wage Average | High correlated with Average Weekly Wage | Discard | Keep | Discard | Discard (already kept Average Weekly Wage) |
| Accident_ymd | High correlated with Assembly_ymd and C2_ymd | Keep | Discard | Discard | Discard |
| Assembly_ymd | High correlated with Accident_ymd and C2_ymd | Keep | Discard | Keep | Discard |
| C2_ymd | High correlated with Accident_ymd and Assembly_ymd | Discard | Discard | Keep | Discard |

In [ ]:
include_variables_ar = [
    "Attorney/Representative",
    "Average Weekly Wage",
    "Carrier Name",
    "Carrier Type",
    "District Name",
    "IME-4 Count",
    "Industry Code",
    "Medical Fee Region",
    "WCIO Nature of Injury Code",
    "Zip Code",
    "C-3 Received",
    "First Hearing Happened",
    "Injury Day of Week"
]

## Model Comparison and Selection

In [ ]:
X_train_ar = X_train_ar[include_variables_ar]
X_val_ar = X_val_ar[include_variables_ar]

### RandomForestClassifier

In [ ]:
results_rf_ar = validation_rf(X_train_ar, y_train_ar, X_val_ar, y_val_ar)

In [ ]:
results_rf_ar

### HistGradientBoostingClassifier

In [ ]:
results_hgb_ar = validation_hgb(X_train_ar, y_train_ar, X_val_ar, y_val_ar)

In [ ]:
results_hgb_ar

### MLPClassifier

In [ ]:
results_mlp_ar = validation_mlp(X_train_ar, y_train_ar, X_val_ar, y_val_ar)

In [ ]:
results_mlp_ar

### StackingClassifier

In [ ]:
results_stack_ar = validation_stack(X_train_ar, y_train_ar, X_val_ar, y_val_ar)

In [ ]:
results_stack_ar

### Comparing Models

In [ ]:
def create_summary_table_ar(results_rf_ar, results_hgb_ar, results_mlp_ar, results_stack_ar):
    results = []
    models = [
        ("Random Forest", results_rf_ar),
        ("Hist Gradient Boosting", results_hgb_ar),
        ("MLP Classifier", results_mlp_ar),
        ("Stacking Classifier", results_stack_ar)
    ]

    for model_name, model_results in models:
        train_f1 = model_results['train_f1']
        val_f1 = model_results['val_f1']
        elapsed_time = model_results['elapsed_time']
        
        results.append({
            "Model": model_name,
            "Train F1 Score": train_f1,
            "Validation F1 Score": val_f1,
            "Time Taken (s)": elapsed_time
        })
    
    summary_table = pd.DataFrame(results)
    
    return summary_table

summary_table_ar = create_summary_table_ar(results_rf_ar, results_hgb_ar, results_mlp_ar, results_stack_ar)
summary_table_ar

All the models showed strong performance, with high F1 scores. The **StackingClassifier** slightly outperformed the others, achieving the best training and validation results. For that reason, we selected the **StackingClassifier** as our final model to predict the **'Agreement Reached'** column.

## Final Agreement Reached Model

In [ ]:
rf_model = RandomForestClassifier(
        n_estimators=50, 
        max_depth=9, 
        min_samples_split=50, 
        random_state=15
    )
    
hgb_model = HistGradientBoostingClassifier(
        max_iter=2000, 
        max_depth=10, 
        random_state=15
    )
    
estimators = [
        ('rf', rf_model),
        ('hgbc', hgb_model)
    ]
    
final_estimator = MLPClassifier(
        solver='adam',
        hidden_layer_sizes=(X_train.shape[1] * 2, X_train.shape[1] * 2, X_train.shape[1] * 2),
        activation='relu',
        learning_rate_init=0.0025,
        max_iter=1000,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=20,
        tol=1e-6
    )
    
final_model_ar = StackingClassifier(
        estimators=estimators,
        final_estimator=final_estimator
    )


final_model_ar.fit(X_train_ar, y_train_ar)

# Applying the Agreement Reached Model

In [ ]:
X_test_ar = df_test_ar[include_variables_ar]  

predictions = final_model_ar.predict(X_test_ar)

df_test_enc_scal['Agreement Reached'] = predictions

# Applying the Claim Injury Type Model

In [ ]:
df_test_model = df_test_enc_scal[include_variables]

In [ ]:
df_test_model.info()

In [ ]:
X_train_enc_scal[include_variables].info()

In [ ]:
df_test_model['Agreement Reached'] = df_test_model['Agreement Reached'].astype('float64')

In [ ]:
encoded_predictions = final_model.predict(df_test_model)

In [ ]:
label_path = 'label_encoder.pkl'
label_encoder = joblib.load(label_path)

In [ ]:
df_test_enc_scal['Claim_Injury_Type'] = label_encoder.inverse_transform(encoded_predictions)

In [ ]:
df_test_enc_scal['Claim_Injury_Type'].head()

In [ ]:
df_test_enc_scal['Claim_Injury_Type'].value_counts()

In [ ]:
df_test_indexes = pd.read_csv('df_test_indexes.csv')

In [ ]:
df_test_indexes.head()

In [ ]:
df_test_indexes = df_test_indexes['Claim Identifier']

In [ ]:
submission_df = pd.DataFrame({
    'Claim Identifier': df_test_indexes,  
    'Claim Injury Type': df_test_enc_scal['Claim_Injury_Type']
})

submission_df.to_csv('submission.csv', index=False)

In [ ]:
joblib.dump(final_model, 'final_model.pkl')
print("Model saved as 'final_model.pkl'")